# Day 3 — 股价崩盘标签定义

**论文对应**: 标签构造 → 未来20日最大回撤(MDD)超过阈值标记为崩盘

**修复内容**:
- 显式 `min_periods=20` 参数，避免隐式行为
- 增加多阈值标签 (10%/15%/20%) 用于稳健性检验
- 增加标签分布可视化和时序统计

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# ==========================================
# 第1步: 读取原始价格数据
# ==========================================
df = pd.read_csv(r"C:\Users\1\Desktop\项目\stock-data\Prices_fixed.csv")
df["Date"] = pd.to_datetime(df["Date"])

print("原始数据:", df.shape)
print("日期范围:", df["Date"].min(), "~", df["Date"].max())
print("股票:", [c for c in df.columns if c != "Date"])

In [ ]:
# ==========================================
# 第2步: 宽表 -> 长表
# ==========================================
df_long = df.melt(
    id_vars="Date",
    var_name="symbol",
    value_name="close"
).sort_values(["symbol", "Date"]).reset_index(drop=True)

print("长表维度:", df_long.shape)

In [ ]:
# ==========================================
# 第3步: 计算日收益率 (t-1 到 t 的收益率)
# ==========================================
df_long["return"] = df_long.groupby("symbol")["close"].pct_change()

In [ ]:
# ==========================================
# 第4步: 未来20日最大回撤 (MDD) — 标签核心
#
# 定义: 未来20个交易日内, 最低收盘价相对今日下跌的百分比
# MDD(t) = min(close[t+1:t+20]) / close[t] - 1
# 若 MDD < -10%, 则crash=1
#
# 使用 shift(-1) 排除当日, rolling(20, min_periods=20) 确保20日窗口完整
# ==========================================

def compute_future_mdd(close_series, horizon=20):
    """计算未来 horizon 日最大回撤"""
    # 未来 horizon 日的最低价 (从t+1开始, 共horizon天)
    future_min = close_series.shift(-1).rolling(horizon, min_periods=horizon).min()
    return future_min / close_series - 1.0

# 按股票分组计算
df_long["future_mdd_20"] = (
    df_long.groupby("symbol")["close"]
    .transform(lambda x: compute_future_mdd(x, horizon=20))
)

print("MDD 计算完成")
print("有效MDD行数:", df_long["future_mdd_20"].notna().sum())
print("NaN行数 (每只股票最后20天):", df_long["future_mdd_20"].isna().sum())

In [ ]:
# ==========================================
# 第5步: 多阈值崩盘标签
# 论文使用10%阈值; 同时构造15%/20%用于稳健性检验
# ==========================================

for threshold, col_name in [(-0.10, "crash_10"), (-0.15, "crash_15"), (-0.20, "crash_20")]:
    df_long[col_name] = np.where(
        df_long["future_mdd_20"] < threshold, 1, 0
    )

# 默认标签: crash = crash_10 (与论文一致)
df_long["crash"] = df_long["crash_10"]

print("标签分布:")
for col in ["crash_10", "crash_15", "crash_20"]:
    counts = df_long[col].value_counts()
    pct = counts[1] / (counts[0] + counts[1]) * 100 if 1 in counts.index else 0
    print(f"  {col}: crash=1 共 {counts.get(1, 0)} 个 ({pct:.1f}%), crash=0 共 {counts.get(0, 0)} 个")

In [ ]:
# ==========================================
# 第6步: 各股票crash次数统计
# ==========================================
crash_by_symbol = (
    df_long.groupby("symbol")["crash"]
    .agg(["sum", "count"])
    .assign(pct=lambda x: x["sum"] / x["count"] * 100)
    .sort_values("sum", ascending=False)
)
crash_by_symbol.columns = ["crash次数", "总样本", "crash占比%"]
print(crash_by_symbol)

In [ ]:
# ==========================================
# 第7步: 可视化 — 标签分布 & 时序热力
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: 各股票crash占比
crash_by_symbol["crash占比%"].plot(kind="barh", ax=axes[0], color="coral")
axes[0].set_title("各股票Crash占比 (%)")
axes[0].set_xlabel("Crash占比 (%)")

# 右: 时序crash事件分布
crash_ts = df_long.groupby("Date")["crash"].mean() * 100
axes[1].fill_between(crash_ts.index, crash_ts.values, alpha=0.5, color="red")
axes[1].plot(crash_ts.index, crash_ts.values, color="darkred", lw=0.5)
axes[1].set_title("每日Crash比例 (10只股票中)")
axes[1].set_ylabel("Crash占比 (%)")
axes[1].axhline(y=14.1, color="gray", ls="--", alpha=0.5, label="均值")
axes[1].legend()

plt.tight_layout()
plt.savefig(r"C:\Users\1\Desktop\项目\stock-data\day3_label_distribution.png", dpi=150)
plt.show()

In [ ]:
# ==========================================
# 第8步: 保存结果
# ==========================================
output_cols = ["Date", "symbol", "close", "return", "future_mdd_20",
               "crash", "crash_10", "crash_15", "crash_20"]

df_long[output_cols].to_csv(
    r"C:\Users\1\Desktop\项目\stock-data\day3_dataset_mdd.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Day3 完成, 已保存 day3_dataset_mdd.csv")
print(f"输出列: {output_cols}")